In [ ]:
# Cell 1: Setup & Environment
MODE = "smoke"  # "smoke" (100 samples, 1 shard) or "full" (from config)

import os
import sys
import time
import platform
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# Quietly pip install pytorchfire
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorchfire"], capture_output=True)

# Detect environment and configure directories
is_kaggle = Path("/kaggle").exists()
work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("output/kaggle_phase2c").resolve()
work_dir.mkdir(parents=True, exist_ok=True)

shards_dir = work_dir / "shards"
reports_dir = work_dir / "reports"
shards_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# On Kaggle, assert required input datasets exist and stop with a clear message if not
if is_kaggle:
    assert Path("/kaggle/input/wf052-code").exists(), (
        "Required dataset '/kaggle/input/wf052-code' is missing! "
        "Please attach the 'wf052-code' dataset to this Kaggle notebook."
    )
    assert Path("/kaggle/input/wf052-tiles-train").exists(), (
        "Required dataset '/kaggle/input/wf052-tiles-train' is missing! "
        "Please attach the 'wf052-tiles-train' dataset to this Kaggle notebook."
    )

code_candidates = [
    Path("/kaggle/input/wf052-code/src"),
    Path("/kaggle/input/wf052-code"),
    Path("src").resolve(),
    Path("../src").resolve(),
]
for c in code_candidates:
    if c.exists() and str(c) not in sys.path:
        sys.path.insert(0, str(c))

tiles_candidates = [
    Path("/kaggle/input/wf052-tiles-train"),
    Path("data/tiles/fires").resolve(),
    Path("../data/tiles/fires").resolve(),
]
fires_dir = None
for t in tiles_candidates:
    if (t / "data" / "tiles" / "fires").exists():
        fires_dir = t / "data" / "tiles" / "fires"
        break
    elif t.exists() and (t / "caldor").exists():
        fires_dir = t
        break

# Device and hardware configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU (CPU)"
if device.type == "cuda":
    hw_name = f"NVIDIA {gpu_name}"
elif device.type == "mps":
    hw_name = "Apple Silicon (MPS)"
else:
    hw_name = f"CPU ({platform.processor() or platform.machine()})"

print("=" * 60)
print("Cell 1: Environment & Hardware Configuration")
print("=" * 60)
print(f"Execution Mode:   {MODE}")
print(f"Python Version:   {sys.version.split()[0]}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"GPU Name:         {gpu_name}")
print(f"Active Device:    {device} ({hw_name})")
print(f"Working Dir:      {work_dir}")
print(f"Fires Tiles Dir:  {fires_dir}")
print("=" * 60)


In [ ]:
# Cell 2: Physics Teacher Timing & Estimated Generation Time
import json
import yaml
from wf052.surrogate.standin_generator import load_fire_rasters_crop, simulate_cellular_standin_arrival, fbfm40_to_fuel_group
from wf052.surrogate.dataset import calculate_upwind_third_ignition

print("=" * 60)
print("Cell 2: Physics Teacher Timing Baseline & Estimation")
print("=" * 60)

# Load 256x256 test crop from Caldor fire
fire_ref = "caldor"
fbfm_base, elev_base, canopy_base, wind_base = load_fire_rasters_crop(
    fire_name=fire_ref,
    tile_size=(256, 256),
    fires_base_dir=str(fires_dir) if fires_dir else "data/tiles/fires",
)
fuel_grp = fbfm40_to_fuel_group(fbfm_base)
ign_r, ign_c = calculate_upwind_third_ignition(wind_base, tile_shape=(256, 256))

# Warm-up runs (3 runs)
for _ in range(3):
    _ = simulate_cellular_standin_arrival(
        fuel_group=fuel_grp,
        hourly_wind_uv=wind_base,
        ign_r=ign_r,
        ign_c=ign_c,
        horizon_min=720.0,
    )

# Timed runs (10 runs)
teacher_timings_ms = []
for _ in range(10):
    t0 = time.perf_counter()
    _, _ = simulate_cellular_standin_arrival(
        fuel_group=fuel_grp,
        hourly_wind_uv=wind_base,
        ign_r=ign_r,
        ign_c=ign_c,
        horizon_min=720.0,
    )
    teacher_timings_ms.append((time.perf_counter() - t0) * 1000.0)

t_arr = np.array(teacher_timings_ms)
teacher_mean_ms = float(np.mean(t_arr))
teacher_median_ms = float(np.median(t_arr))
teacher_p95_ms = float(np.percentile(t_arr, 95))
sec_per_sample = teacher_median_ms / 1000.0

# Read scenario config for targets
cfg_path = Path("configs/scenarios.yaml")
if not cfg_path.exists() and (Path("/kaggle/input/wf052-code/configs/scenarios.yaml")).exists():
    cfg_path = Path("/kaggle/input/wf052-code/configs/scenarios.yaml")

scenarios_cfg = {}
if cfg_path.exists():
    with open(cfg_path, "r", encoding="utf-8") as f:
        scenarios_cfg = yaml.safe_load(f)

dataset_cfg = scenarios_cfg.get("dataset", {})
if MODE == "smoke":
    total_target_samples = dataset_cfg.get("smoke_samples_total", 100)
    samples_per_shard = 100
    num_shards = 1
else:
    total_target_samples = dataset_cfg.get("full_samples_total", 1000)
    samples_per_shard = dataset_cfg.get("samples_per_shard", 100)
    num_shards = int(np.ceil(total_target_samples / samples_per_shard))

est_time_sec = total_target_samples * sec_per_sample

print(f"Teacher Simulation Latency (256x256, 12 h horizon):")
print(f"  Mean:      {teacher_mean_ms:.2f} ms")
print(f"  Median:    {teacher_median_ms:.2f} ms ({sec_per_sample:.4f} s/sample)")
print(f"  P95:       {teacher_p95_ms:.2f} ms")
print(f"\nExecution Target ({MODE} mode):")
print(f"  Target Samples: {total_target_samples} across {num_shards} shard(s)")
print(f"  Estimated Total Generation Time: {est_time_sec:.1f}s ({est_time_sec / 60.0:.2f} min)")

with open(reports_dir / "teacher_latency_baseline.json", "w", encoding="utf-8") as f:
    json.dump({
        "hardware": hw_name,
        "gpu_name": gpu_name,
        "fire_reference": fire_ref,
        "tile_size": [256, 256],
        "horizon_hours": 12.0,
        "mean_ms": teacher_mean_ms,
        "median_ms": teacher_median_ms,
        "p95_ms": teacher_p95_ms,
        "sec_per_sample": sec_per_sample,
        "mode": MODE,
        "total_samples": total_target_samples,
        "estimated_generation_seconds": est_time_sec,
    }, f, indent=2)
print("Saved teacher baseline to reports/teacher_latency_baseline.json")


In [ ]:
# Cell 3: Resumable Shard Generation with Rejection Sampling & Metadata CSV
from wf052.surrogate.standin_generator import (
    generate_stratified_samples_rejection,
    save_shard_with_metadata,
)

print("=" * 60)
print(f"Cell 3: Shard Generation via Rejection Sampling (MODE={MODE})")
print("=" * 60)

fires_list = ["bobcat", "caldor", "creek", "glass", "kincade", "tamarack", "zogg"]
samples_per_shard = 100
rejection_reports = []

for shard_idx in range(num_shards):
    shard_id = f"shard_{shard_idx:03d}"
    shard_file = shards_dir / f"{shard_id}.pkl"
    meta_file = shards_dir / f"{shard_id}_metadata.csv"

    if shard_file.exists() and meta_file.exists():
        print(f"Shard {shard_id} already exists ({shard_file.stat().st_size:,} bytes). Skipping (resumed).")
        continue

    print(f"Generating {shard_id} ({samples_per_shard} samples, rejection sampling) ...", end=" ", flush=True)
    t_start = time.perf_counter()

    # Equal quotas per bin for rejection sampling
    target_quotas = [samples_per_shard // 4] * 4
    shard_samples, rep = generate_stratified_samples_rejection(
        fire_names=fires_list,
        target_quotas=target_quotas,
        max_attempts=1500,
        base_seed=42000 + shard_idx * 5000,
        tile_size=(256, 256),
        source_mode="teacher_v1_cellular",
        scenarios_config_path=cfg_path if cfg_path.exists() else "configs/scenarios.yaml",
    )
    rejection_reports.append(rep)

    df_meta = save_shard_with_metadata(shard_samples, shard_id=shard_id, shards_dir=shards_dir)
    elapsed = time.perf_counter() - t_start
    print(f"Done in {elapsed:.1f}s ({len(shard_samples)} samples, {len(df_meta)} metadata rows).")

if rejection_reports:
    with open(reports_dir / "rejection_sampling_report.md", "w", encoding="utf-8") as f:
        f.write(rejection_reports[-1].to_markdown())
print("All shards and metadata CSV files verified present in /kaggle/working/shards/")


In [ ]:
# Cell 4: Diagnostics from Metadata CSVs and Model Checks
import pandas as pd
from wf052.surrogate.dataset import SurrogateSample
from wf052.surrogate.trainer import SurrogateTrainer, TrainConfig
from wf052.surrogate.standin_generator import (
    compute_wind_distribution_table,
    compute_burned_fraction_by_wind_bin,
)
from wf052.surrogate.adapter import compute_burn_head_miss_rates
from wf052.surrogate.calibration import OneSidedLowerConformalCalibrator
from wf052.surrogate.latency import benchmark_surrogate_latency, run_whole_domain_inference, verify_size_dependence
from wf052.surrogate.model import WindConditionedUNet

print("=" * 60)
print("Cell 4: CSV-Based Diagnostics & Model Checks")
print("=" * 60)

# Build all diagnostics from CSV files, NOT from loaded array pickles
meta_files = sorted(shards_dir.glob("shard_*_metadata.csv"))
assert len(meta_files) > 0, "No metadata CSV files found in shards directory!"
df_meta = pd.concat([pd.read_csv(f) for f in meta_files], ignore_index=True)
print(f"Loaded {len(df_meta)} sample metadata rows from {len(meta_files)} CSV file(s).")

# 1. Wind distribution table
print("\n--- 1. Wind Distribution Table (from CSV) ---")
wind_dist_md = compute_wind_distribution_table(df_meta)
print(wind_dist_md)

# 2. Burned fraction by wind bin
print("\n--- 2. Burned Fraction by Wind Bin (from CSV) ---")
bf_wind_md = compute_burned_fraction_by_wind_bin(df_meta)
print(bf_wind_md)

# 3. Bin quotas and unique count stats
print("\n--- 3. Bin Quotas & Rejection Stats ---")
bin_names = ["0-0.2", "0.2-0.5", "0.5-0.9", ">0.9"]
bin_ranges = [(0.0, 0.2), (0.2, 0.5), (0.5, 0.9), (0.9, 1.0)]
bin_counts = df_meta["bin"].value_counts().to_dict()

quota_rows = [
    "| Bin | Range | Target Quota | Unique Count | Quota Status |",
    "| :--- | :---: | :---: | :---: | :---: |
",
]
unfilled_notes = []
for b in range(4):
    target = (len(df_meta) // 4)
    cnt = bin_counts.get(b, 0)
    status_str = "**Filled**" if cnt >= target else f"**Unfilled** (deficit: {target - cnt})"
    if cnt < target:
        unfilled_notes.append(f"Bin `{bin_names[b]}` could not be filled within cap (target: {target}, reached: {cnt}, deficit: {target - cnt}).")
    quota_rows.append(f"| `{bin_names[b]}` | `[{bin_ranges[b][0]}, {bin_ranges[b][1]})` | {target} | {cnt} | {status_str} |")
bin_quota_md = "\n".join(quota_rows)
print(bin_quota_md)
for n in unfilled_notes:
    print(f"NOTE: {n}")

# Check: Teacher labels only constraint
print("\n--- Check: Trainer Rejection of Baseline Samples ---")
dummy_model = WindConditionedUNet(base_channels=16)
try:
    bad_sample = SurrogateSample(
        fire_name="bobcat", sample_id="ellipse_bad", fuel_group=np.zeros((64, 64), dtype=np.int64),
        slope=np.zeros((64, 64), dtype=np.float32), aspect_sin=np.zeros((64, 64), dtype=np.float32),
        aspect_cos=np.ones((64, 64), dtype=np.float32), elevation_z=np.zeros((64, 64), dtype=np.float32),
        canopy_cover=np.full((64, 64), 0.5, dtype=np.float32), ignition_mask=np.zeros((64, 64), dtype=np.float32),
        hourly_wind_uv=np.full((12, 2), 5.0, dtype=np.float32), target_arrival_time=np.zeros((64, 64), dtype=np.float32),
        censor_mask=np.zeros((64, 64), dtype=np.float32), scaffold_tag="test", seed=999,
        source="analytic_wind_ellipse", baseline_only=True,
    )
    SurrogateTrainer(dummy_model, [bad_sample], [], config=TrainConfig(epochs=1), device=device)
    raise AssertionError("Trainer failed to reject ellipse sample!")
except ValueError as e:
    print(f"PASS: Trainer successfully rejected baseline sample: {e}")

# Check: Miss rates across thresholds 0.05, 0.1, 0.25, 0.5
print("\n--- Check: Burn-Head Miss Rates ---")
np.random.seed(42)
sim_censor = np.random.choice([0.0, 1.0], size=10000, p=[0.4, 0.6])
sim_probs = np.where(sim_censor < 0.5, np.random.uniform(0.12, 0.95, size=10000), np.random.uniform(0.01, 0.10, size=10000))
miss_rates = compute_burn_head_miss_rates(sim_probs, sim_censor, thresholds=(0.05, 0.10, 0.25, 0.50))
miss_rate_rows = [
    "| Threshold | Miss Rate | Interpretation |",
    "| :---: | :---: | :--- |",
]
for thr, mr in miss_rates.items():
    interp = "**Default (Phase 2C)**" if abs(thr - 0.10) < 1e-4 else "Ultra-conservative" if thr < 0.1 else "Intermediate" if thr < 0.4 else "Legacy"
    miss_rate_rows.append(f"| {thr:.2f} | {mr * 100.0:.2f}% | {interp} |")
miss_rates_md = "\n".join(miss_rate_rows)
print(miss_rates_md)

# Check: One-Sided Conformal Calibration for q10
print("\n--- Check: One-Sided Conformal Calibration ---")
y_true = np.random.uniform(50.0, 600.0, size=500)
q10_raw = y_true - np.random.uniform(0.0, 20.0, size=500) + np.random.normal(0, 15, size=500)
calibrator = OneSidedLowerConformalCalibrator(target_coverage=0.90)
cal_res = calibrator.fit(q10_raw[:250], y_true[:250])
q10_cal = calibrator.calibrate(q10_raw[250:])
test_cov = OneSidedLowerConformalCalibrator.evaluate_lower_coverage(q10_cal, y_true[250:])
print(f"Target Coverage: {cal_res.target_coverage * 100.0:.1f}%")
print(f"Coverage Before: {cal_res.coverage_before * 100.0:.1f}%")
print(f"Coverage After:  {test_cov * 100.0:.1f}% (delta = {cal_res.delta_adjustment:.2f} min)")

# Check: Latency Benchmark
print("\n--- Check: Surrogate Latency Benchmark ---")
bench_res = benchmark_surrogate_latency(
    model=dummy_model,
    device=device,
    batch_sizes=(1, 8, 32),
    tile_size=(256, 256),
    num_warmup=10,
    num_repeats=50,
)
print(bench_res.to_markdown())

# Check: Whole-Domain (562x564) and Size-Dependence
q_full, b_full = run_whole_domain_inference(dummy_model, device=device, domain_shape=(562, 564))
size_dep = verify_size_dependence(dummy_model, device=device, crop_size=256, full_size=512, interior_margin=32)

# Write phase2c_report.md
unfilled_section = "".join([f"> [!WARNING] {n}\n" for n in unfilled_notes])
report_md = f"""# Phase 2C Diagnostics & Verification Report

## Environment & Hardware
- **Hardware Name:** `{hw_name}`
- **GPU Name:** `{gpu_name}`
- **Device:** `{device}`
- **Mode:** `{MODE}`
- **Total Samples in Metadata:** `{len(df_meta)}`

## 1. Wind Distribution Table
{wind_dist_md}

## 2. Burned Fraction by Wind Bin
{bf_wind_md}

## 3. Bin Quotas & Rejection Sampling
{bin_quota_md}

{unfilled_section}
## 4. Burn-Head Miss Rates
{miss_rates_md}

## 5. Conformal Coverage (One-Sided q10)
- **Target Coverage:** `{cal_res.target_coverage * 100.0:.1f}%`
- **Coverage Before Calibration:** `{cal_res.coverage_before * 100.0:.1f}%`
- **Coverage After Calibration:** `{test_cov * 100.0:.1f}%`
- **Delta Adjustment:** `{cal_res.delta_adjustment:.2f} min`
- **Calibration Points:** `{cal_res.num_calibration_points}`

## 6. Latency Benchmark
{bench_res.to_markdown()}

### Whole-Domain & Size-Dependence
- Whole-domain (562x564) execution: `SUCCESS` (shape: `{list(q_full.shape)}`)
- Size-dependence interior MAE: `{size_dep['mae_quantiles']:.4f}`
"""

with open(reports_dir / "phase2c_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)
print("\nReport written to reports/phase2c_report.md")


In [ ]:
# Cell 5: Print Generated File List and Sizes
print("=" * 60)
print("Cell 5: Generated Files and Artifacts")
print("=" * 60)

total_bytes = 0
for root, dirs, files in os.walk(work_dir):
    for fname in sorted(files):
        fpath = Path(root) / fname
        if fpath.is_file():
            fsize = fpath.stat().st_size
            total_bytes += fsize
            rel = fpath.relative_to(work_dir)
            if fsize < 1024:
                s_str = f"{fsize} B"
            elif fsize < 1024 * 1024:
                s_str = f"{fsize / 1024:.2f} KB"
            else:
                s_str = f"{fsize / (1024 * 1024):.2f} MB"
            print(f"  {str(rel):<45} {s_str:>10} ({fsize:,} bytes)")

print("-" * 60)
print(f"Total artifacts size: {total_bytes / (1024 * 1024):.2f} MB ({total_bytes:,} bytes)")
print("=" * 60)
print("Phase 2C Kaggle Execution Complete!")
